# Prisoner's Dilemma: classical and quantum strategies

Research companion to Esther Cui's *Simulating Game Theory and Strategic Interactions Using Quantum Computing*.

This walkthrough builds the classical baseline for two to four players, then examines how Q changes best responses in the two-player Eisert–Wilkens–Lewenstein (EWL) game. Each strategy is fixed; the equilibrium calculation concerns pure-strategy Nash equilibria.

From the repository root, install `python -m pip install -e '.[notebook]'`. The calculations below need NumPy and the installed package. See [methods](../docs/methods.md) for the matrices and conventions.

In [1]:
from itertools import product
from math import pi

import numpy as np
from IPython.display import Markdown, display

from quantum_pd import classical_nash, classical_payoff, pure_nash_mask
from quantum_pd.ewl import C, D, Q, expected_payoff

np.set_printoptions(precision=6, suppress=True)

## Classical payoff rules

C is encoded as 0 and D as 1. A payoff vector is ordered by player. For more than two players, `collective` uses the all-C / all-D / mixed-profile rule; `pairwise` sums payoffs against all opponents.

In [2]:
for actions in product(range(2), repeat=2):
    print(actions, "→", classical_payoff(actions).tolist())

rows = ["| Players | Rule | Equilibrium actions | Payoffs |", "|---|---|---|---|"]
for players in (2, 3, 4):
    for rule in ("collective", "pairwise"):
        result = classical_nash(players, model=rule)[0]
        rows.append(f"| {players} | {rule} | {result['actions']} | {result['payoffs']} |")
display(Markdown("\n".join(rows)))

(0, 0) → [3.0, 3.0]
(0, 1) → [0.0, 5.0]
(1, 0) → [5.0, 0.0]
(1, 1) → [1.0, 1.0]


| Players | Rule | Equilibrium actions | Payoffs |
|---|---|---|---|
| 2 | collective | [1, 1] | [1.0, 1.0] |
| 2 | pairwise | [1, 1] | [1.0, 1.0] |
| 3 | collective | [1, 1, 1] | [1.0, 1.0, 1.0] |
| 3 | pairwise | [1, 1, 1] | [2.0, 2.0, 2.0] |
| 4 | collective | [1, 1, 1, 1] | [1.0, 1.0, 1.0, 1.0] |
| 4 | pairwise | [1, 1, 1, 1] | [3.0, 3.0, 3.0, 3.0] |

## Q against defection

The EWL game applies $J$, one strategy per player, then $J^\dagger$ before measurement. At maximal entanglement $\gamma=\pi/2$, the named strategies are $C=I$, $D=iY$, and $Q=iZ$.

Q differs from the classical action C even though `(Q, Q)` yields the cooperative outcome. In particular, `(C, D)` pays `(0, 5)`, while `(Q, D)` pays `(5, 0)`.

In [3]:
profiles = [("C, D", [C, D]), ("Q, D", [Q, D]), ("D, Q", [D, Q]), ("Q, Q", [Q, Q])]
rows = ["| Strategies | Expected payoffs |", "|---|---|"]
for label, profile in profiles:
    rows.append(f"| {label} | {expected_payoff(profile).round(6).tolist()} |")
display(Markdown("\n".join(rows)))

| Strategies | Expected payoffs |
|---|---|
| C, D | [0.0, 5.0] |
| Q, D | [5.0, 0.0] |
| D, Q | [0.0, 5.0] |
| Q, Q | [3.0, 3.0] |

## Enumerate the complete two-player strategy grid

Each player chooses one of 15 theta values and 8 phi values, including endpoints. There are 120 strategies per player and 14,400 joint profiles. All phase values are included. `pure_nash_mask` compares each player's payoff with every unilateral alternative and retains all ties within an absolute tolerance of $10^{-10}$.

In [4]:
grid = list(product(np.linspace(0, pi, 15), np.linspace(0, pi / 2, 8)))
payoffs = np.array([[expected_payoff([a, b]) for b in grid] for a in grid])
equilibria = np.argwhere(pure_nash_mask(payoffs, atol=1e-10))
print(f"Strategies per player: {len(grid)}; joint profiles: {len(grid) ** 2:,}")
for a, b in equilibria:
    print("Angles (theta/pi, phi/pi):", np.array([grid[a], grid[b]]) / pi)
    print("Payoffs:", payoffs[a, b])
assert equilibria.tolist() == [[7, 7]]
assert np.allclose(grid[7], Q)
assert np.allclose(payoffs[7, 7], [3, 3])

Strategies per player: 120; joint profiles: 14,400
Angles (theta/pi, phi/pi): [[0.  0.5]
 [0.  0.5]]
Payoffs: [3. 3.]


## Why Q, Q is stable throughout the allowed angle domain

When the other player holds Q, the EWL payoff formula is

$$u(U(\theta,\phi),Q)=\cos^2(\theta/2)(3\sin^2\phi+\cos^2\phi)\leq 3.$$

Thus no unilateral choice in the restricted EWL family improves on the payoff 3 at Q. This analytical bound extends beyond the finite grid. The code below checks the formula against the circuit calculation at every grid strategy.

In [5]:
deviation_payoffs = np.array([expected_payoff([angle, Q])[0] for angle in grid])
formula = np.array(
    [np.cos(theta / 2) ** 2 * (3 * np.sin(phi) ** 2 + np.cos(phi) ** 2) for theta, phi in grid]
)
assert np.allclose(deviation_payoffs, formula, rtol=0, atol=1e-12)
print("Payoff at Q, Q:", expected_payoff([Q, Q]))
print("Best unilateral payoff on the grid:", round(float(deviation_payoffs.max()), 12))

Payoff at Q, Q: [3. 3.]
Best unilateral payoff on the grid: 3.0


## Continue the study

The EWL calculation above concerns two players and its stated strategy family. For the multiplayer circuit benchmark, see the [ZZ circuit reference](../docs/zz-reference.md). The [second notebook](02_circuit_experiments.ipynb) explores variational payoffs and coordination. The [archive](../archive/README.md) retains the original two- to four-player research notebooks.

Reference: [Eisert, Wilkens, and Lewenstein, *Quantum Games and Quantum Strategies*](https://arxiv.org/abs/quant-ph/9806088), equations (3), (5), (7), and (8), version 4.